# Theatrical Plays

vorpal's play mode assigns a distinct narrator voice to each character in a stage play. Feed it *Hamlet* and each character speaks in their own voice. The casting is automatic — the algorithm reads gender and role weight from the text and assigns from the voice registry — but every assignment can be overridden.

This notebook covers:
- Fetching a Gutenberg play
- Inspecting the automatic cast sheet
- Building the audiobook with `vorpal play`
- Overriding specific voice assignments
- Auditioning the cast before committing to a full build

**Prerequisites:** vorpal installed (`pip install -e .`), GPU recommended for full synthesis.

## Requirements

- `vorpal` installed in the active environment
- `ffmpeg` on PATH
- For fast preview: `--draft` flag (Piper CPU engine, no GPU needed)
- For full quality: Kokoro + GPU

In [ ]:
import subprocess
import json
import pathlib

from vorpal.play.models import PlayDoc
from vorpal.play.casting import build_cast
from vorpal.tts.voices import VOICE_REGISTRY

print(f"{len(VOICE_REGISTRY)} voices available for casting")

## Fetching a play from Project Gutenberg

`vorpal fetch-play` downloads the plain-text version of a Gutenberg play.
Pass the Gutenberg ID or a title (fuzzy match). Some useful IDs:

| ID | Title |
|----|-------|
| 1787 | Hamlet — Shakespeare |
| 2265 | A Midsummer Night's Dream — Shakespeare |
| 1145 | Waiting for Godot — Beckett |
| 2082 | Othello — Shakespeare |

In [ ]:
result = subprocess.run(
    ["vorpal", "fetch-play", "--title-or-id", "1787", "--corpus-dir", "plays"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)
print(f"Return code: {result.returncode}")

play_path = next(pathlib.Path("plays").glob("*.txt"), None)
print(f"\nPlay file: {play_path}")

## Inspecting the automatic cast

`vorpal cast` prints the voice assignment for each character without building anything.
Review it before committing to a full synthesis run.

In [ ]:
result = subprocess.run(
    ["vorpal", "cast", str(play_path)],
    capture_output=True, text=True
)
print(result.stdout)

## Auditioning the cast

Before a full build, generate a short WAV for each character to hear the actual assignments.
Each file is named `<CHARACTER>_<voice_id>.wav`.

In [ ]:
result = subprocess.run(
    ["vorpal", "cast-audition", str(play_path), "--output", "cast_audition"],
    capture_output=True, text=True
)
print(result.stdout)
print(f"Return code: {result.returncode}")

audition_files = list(pathlib.Path("cast_audition").glob("*.wav"))
print(f"\nGenerated {len(audition_files)} audition clips:")
for f in sorted(audition_files):
    print(f"  {f.name}")

## Building the play audiobook

We use `--draft` here so it runs fast on CPU. Remove `--draft` for production quality.

`--chapters act` creates one chapter per act (default). Use `--chapters scene` for
finer navigation.

In [ ]:
result = subprocess.run(
    [
        "vorpal", "play", str(play_path),
        "--chapters", "act",
        "--stage-directions", "narrator",
        "--draft",
        "--output", "hamlet_draft",
    ],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
print(f"Return code: {result.returncode}")

In [ ]:
m4b = pathlib.Path("hamlet_draft.m4b")
print(f"{m4b.name}: {m4b.stat().st_size / 1e6:.1f} MB")

probe = subprocess.run(
    ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_chapters", str(m4b)],
    capture_output=True, text=True
)
chapters = json.loads(probe.stdout).get("chapters", [])
print(f"Chapters: {len(chapters)}")
for ch in chapters:
    start = float(ch["start_time"])
    h, rem = divmod(int(start), 3600)
    m, s = divmod(rem, 60)
    print(f"  {ch['tags']['title']}  ({h}h{m:02d}m{s:02d}s)")

## Overriding voice assignments

If any casting choice doesn't suit you, override it with `--cast-override`.
The override is a comma-separated string of `CHARACTER=voice_id` pairs.

In [ ]:
# Preview the cast with Hamlet reassigned to blend_deep_steady
result = subprocess.run(
    ["vorpal", "cast", str(play_path), "--cast-override", "HAMLET=blend_deep_steady"],
    capture_output=True, text=True
)
print(result.stdout)

## Next steps

For a full-quality production build, remove `--draft` and let Kokoro run on GPU:

```bash
vorpal play plays/hamlet.txt --chapters act --stage-directions narrator --output hamlet
```

See the CLI reference (`vorpal play --help`) for the full option reference.